# AI Data Quality Monitor — Exploratory Analysis

This notebook is used to:
1. Explore the feature distributions from simulated events
2. Establish baseline statistics for drift detection
3. Visualize drift effects
4. Prototype and validate detection logic before deploying to Spark

In [ ]:
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 4)
print('Libraries loaded ✓')

## 1. Simulate Feature Data

In [ ]:
DEVICE_TYPES = ['mobile', 'desktop', 'tablet']

def generate_normal_events(n=500):
    return pd.DataFrame([
        {
            'user_id': f'user_{random.randint(1, 100_000)}',
            'age': random.randint(18, 75),
            'purchase_amount': round(random.uniform(0.0, 500.0), 2),
            'session_duration': random.randint(30, 3600),
            'page_views': random.randint(1, 50),
            'device_type': random.choice(DEVICE_TYPES),
        }
        for _ in range(n)
    ])

def generate_drifted_events(n=500):
    """Simulates age shift + purchase amount spikes."""
    return pd.DataFrame([
        {
            'user_id': f'user_{random.randint(1, 100_000)}',
            'age': random.randint(65, 110),         # AGE DRIFT
            'purchase_amount': round(random.uniform(5000, 50000), 2),  # AMOUNT SPIKE
            'session_duration': random.randint(30, 3600),
            'page_views': random.randint(1, 50),
            'device_type': random.choice(DEVICE_TYPES + ['smartwatch', 'tv']),  # CAT DRIFT
        }
        for _ in range(n)
    ])

baseline_df = generate_normal_events(500)
drifted_df  = generate_drifted_events(500)

print(f'Baseline shape: {baseline_df.shape}')
print(f'Drifted shape:  {drifted_df.shape}')
baseline_df.head()

## 2. Baseline Feature Statistics

In [ ]:
baseline_df.describe()

## 3. Distribution Visualisation

In [ ]:
numeric_features = ['age', 'purchase_amount', 'session_duration', 'page_views']

fig, axes = plt.subplots(2, len(numeric_features), figsize=(18, 8))

for i, col in enumerate(numeric_features):
    # Baseline
    axes[0, i].hist(baseline_df[col].dropna(), bins=30, color='steelblue', alpha=0.75, edgecolor='white')
    axes[0, i].set_title(f'{col} — Baseline', fontsize=11)
    axes[0, i].set_xlabel(col)
    axes[0, i].set_ylabel('Count')

    # Drifted
    axes[1, i].hist(drifted_df[col].dropna(), bins=30, color='tomato', alpha=0.75, edgecolor='white')
    axes[1, i].set_title(f'{col} — Drifted', fontsize=11)
    axes[1, i].set_xlabel(col)
    axes[1, i].set_ylabel('Count')

plt.suptitle('Feature Distributions: Baseline vs Drifted', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 4. KS Test — Drift Detection

In [ ]:
ks_results = []
for col in numeric_features:
    stat, p = stats.ks_2samp(baseline_df[col].dropna(), drifted_df[col].dropna())
    ks_results.append({
        'feature': col,
        'ks_statistic': round(stat, 4),
        'p_value': round(p, 6),
        'drifted': p < 0.05,
    })

ks_df = pd.DataFrame(ks_results)
ks_df.style.applymap(lambda v: 'background-color: #ffcccc' if v == True else '', subset=['drifted'])

## 5. Population Stability Index (PSI)

In [ ]:
def compute_psi(reference, current, n_bins=10, epsilon=1e-4):
    bins = np.percentile(reference, np.linspace(0, 100, n_bins + 1))
    bins[0], bins[-1] = -np.inf, np.inf
    ref_pct = np.histogram(reference, bins=bins)[0] / len(reference) + epsilon
    cur_pct = np.histogram(current,   bins=bins)[0] / len(current)    + epsilon
    return float(np.sum((cur_pct - ref_pct) * np.log(cur_pct / ref_pct)))

psi_results = []
for col in numeric_features:
    psi = compute_psi(baseline_df[col].dropna().values, drifted_df[col].dropna().values)
    level = 'ALERT' if psi > 0.2 else ('WARNING' if psi > 0.1 else 'OK')
    psi_results.append({'feature': col, 'psi': round(psi, 4), 'level': level})

pd.DataFrame(psi_results)

## 6. Null Rate Analysis

In [ ]:
# Inject nulls into drifted set
drifted_with_nulls = drifted_df.copy()
null_mask = np.random.random(len(drifted_with_nulls)) < 0.15
drifted_with_nulls.loc[null_mask, 'age'] = np.nan

null_comparison = pd.DataFrame({
    'feature': baseline_df.columns,
    'baseline_null_rate': (baseline_df.isnull().mean() * 100).values,
    'current_null_rate': (drifted_with_nulls.isnull().mean() * 100).values,
})
null_comparison['delta_pp'] = (null_comparison['current_null_rate'] - null_comparison['baseline_null_rate']).round(2)
null_comparison['alert'] = null_comparison['delta_pp'].abs() > 5
null_comparison

## 7. Categorical Feature Drift (Chi-Squared)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

baseline_df['device_type'].value_counts(normalize=True).plot.bar(ax=axes[0], color='steelblue', title='device_type — Baseline')
drifted_df['device_type'].value_counts(normalize=True).plot.bar(ax=axes[1], color='tomato', title='device_type — Drifted')

axes[0].set_ylabel('Proportion')
axes[1].set_ylabel('Proportion')
plt.tight_layout()
plt.show()

# Chi-squared test
all_cats = sorted(set(baseline_df['device_type']) | set(drifted_df['device_type']))
observed = [drifted_df['device_type'].value_counts().get(c, 0) for c in all_cats]
expected_freq = baseline_df['device_type'].value_counts(normalize=True)
expected = [expected_freq.get(c, 1e-4) * len(drifted_df) for c in all_cats]

chi2, p = stats.chisquare(f_obs=observed, f_exp=expected)
print(f'Chi2={chi2:.4f}, p={p:.6f}, drifted={p < 0.05}')

## 8. Summary Dashboard

In [ ]:
print('=' * 55)
print('  DATA QUALITY & DRIFT SUMMARY')
print('=' * 55)

print('\n[KS Test Results]')
for r in ks_results:
    status = '⚠ DRIFT' if r['drifted'] else '✓ OK'
    print(f"  {r['feature']:20s} ks={r['ks_statistic']:.4f}  p={r['p_value']:.6f}  {status}")

print('\n[PSI Results]')
for r in psi_results:
    print(f"  {r['feature']:20s} psi={r['psi']:.4f}  [{r['level']}]")

print('=' * 55)